# Multiplication test
A small open model's answers to 30 four-digit multiplications, compared with the real answers digit by digit, and cells to ask it your own sums three ways: answer only, step by step, and with a calculator.

The first part needs nothing but numpy. The model part runs on a free Colab GPU (Runtime → Change runtime type → T4 GPU), or slowly on a CPU. Answers can differ a little on different hardware.

In [1]:
# The 30 four-digit sums the model was asked ("Answer with just the number."), and what it answered.
# (a, b, its answer). The real answer is just a * b.
RUNS = [
    (4331, 2394, 10256874),
    (8453, 9612, 79900856),
    (4865, 9883, 47201845),
    (1274, 3366, 4290864),
    (6263, 9616, 59408256),
    (5861, 2873, 16904843),
    (3640, 1762, 6598680),
    (8227, 7590, 61943330),
    (9729, 8347, 81605183),
    (1311, 2797, 3680547),
    (7185, 9583, 68042555),
    (7191, 4597, 32806533),
    (7542, 2672, 19803304),
    (8315, 3705, 31246925),
    (7285, 2015, 14693225),
    (7558, 7946, 59031268),
    (9005, 3567, 32184555),
    (7994, 7239, 58061866),
    (6583, 5148, 33907224),
    (5516, 2124, 11807904),
    (4359, 6285, 27501585),
    (3237, 9704, 31568708),
    (5275, 2663, 13849045),
    (9693, 9663, 92805449),
    (5118, 2062, 10437996),
    (2270, 6187, 13945490),
    (2992, 2562, 7640832),
    (9129, 9963, 89705447),
    (9135, 5948, 53706260),
    (1983, 8845, 17062435),
]

In [2]:
import numpy as np

right = sum(ans == a * b for a, b, ans in RUNS)
same_len = sum(len(str(ans)) == len(str(a * b)) for a, b, ans in RUNS)
first = sum(str(ans)[0] == str(a * b)[0] for a, b, ans in RUNS)
last = sum(str(ans)[-1] == str(a * b)[-1] for a, b, ans in RUNS)
close = sum(abs(ans - a * b) / (a * b) < 0.01 for a, b, ans in RUNS)
print(f"right: {right} of {len(RUNS)}")
print(f"right number of digits: {same_len} · first digit right: {first} · last digit right: {last}")
print(f"within 1% of the real answer: {close}")

# Digit by digit, for the 8-digit answers: how often each position was right (left to right).
eight = [(str(ans), str(a * b)) for a, b, ans in RUNS if len(str(a * b)) == 8 and len(str(ans)) == 8]
hits = np.array([[s[i] == t[i] for i in range(8)] for s, t in eight])
print(f"\n{len(eight)} eight-digit answers, share right at each position:")
print(np.round(hits.mean(axis=0), 2))

right: 0 of 30
right number of digits: 30 · first digit right: 25 · last digit right: 27
within 1% of the real answer: 16

26 eight-digit answers, share right at each position:
[0.81 0.46 0.12 0.19 0.04 0.19 0.23 0.92]


In [3]:
# The last digit of a product only depends on the last digit of each number:
# last digit of (a × b) = last digit of (last digit of a × last digit of b)
print(all((a * b) % 10 == ((a % 10) * (b % 10)) % 10 for a, b, _ in RUNS))

# The middle digits depend on every digit of both numbers, through the carries.
# Long multiplication, column by column from the right, printing each carry:
def carries(a, b):
    xs, ys = [int(c) for c in str(a)][::-1], [int(c) for c in str(b)][::-1]
    cols = [0] * (len(xs) + len(ys))
    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            cols[i + j] += x * y
    carry, digits = 0, []
    for k, c in enumerate(cols):
        total = c + carry
        digits.append(total % 10)
        print(f"column {k}: sum {c} + carry {carry} = {total} -> write {total % 10}, carry {total // 10}")
        carry = total // 10
    return int("".join(map(str, digits[::-1])))

print(carries(6583, 5148), 6583 * 5148)

True
column 0: sum 24 + carry 0 = 24 -> write 4, carry 2
column 1: sum 76 + carry 2 = 78 -> write 8, carry 7
column 2: sum 75 + carry 7 = 82 -> write 2, carry 8
column 3: sum 91 + carry 8 = 99 -> write 9, carry 9
column 4: sum 69 + carry 9 = 78 -> write 8, carry 7
column 5: sum 31 + carry 7 = 38 -> write 8, carry 3
column 6: sum 30 + carry 3 = 33 -> write 3, carry 3
column 7: sum 0 + carry 3 = 3 -> write 3, carry 0
33889284 33889284


## Ask it yourself

In [4]:
# Install the library that downloads and runs open models (Colab already has numpy and torch).
!pip -q install transformers accelerate

In [5]:
# Load the small open model (1.5 billion numbers inside). The first run downloads it (about 3 GB).
import json, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()

def generate(msgs, max_new, tools=None):
    text = tok.apply_chat_template(msgs, tools=tools, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False)   # always the most likely next piece
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

In [6]:
CALC = {"type": "function", "function": {
    "name": "calculator", "description": "Work out an arithmetic expression exactly, e.g. '4731 * 2986'.",
    "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "The arithmetic to work out"}},
                   "required": ["expression"]}}}

def calculate(expr):
    expr = expr.replace("×", "*").replace("x", "*").replace(",", "")
    if not re.fullmatch(r"[\d\s+\-*/().]+", expr):
        return None
    return eval(expr, {"__builtins__": {}}, {})   # only digits, spaces and + - * / ( ) get this far, so it can only do arithmetic

def answer_only(q):
    return generate([{"role": "system", "content": "Answer with just the number."}, {"role": "user", "content": q}], 30)

def step_by_step(q):
    sys = "Work it out step by step, then give the final answer on the last line as 'Answer: <number>'."
    return generate([{"role": "system", "content": sys}, {"role": "user", "content": q}], 600)

def with_calculator(q):
    msgs = [{"role": "system", "content": "You can use the calculator tool. Give the final answer as a number."}, {"role": "user", "content": q}]
    first = generate(msgs, 80, tools=[CALC])
    m = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", first, re.S)
    if not m:
        return {"used_calculator": False, "final": first}
    expr = json.loads(m.group(1)).get("arguments", {}).get("expression", "")
    result = calculate(expr)
    msgs += [{"role": "assistant", "content": first}, {"role": "tool", "content": json.dumps({"result": result})}]
    return {"used_calculator": True, "expression": expr, "calculator": result, "final": generate(msgs, 80, tools=[CALC])}

In [7]:
a, b = 6583, 5148   # change these to your own sum
q = f"What is {a} × {b}?"
print("real answer:   ", a * b)
print("answer only:   ", answer_only(q))
print("\nstep by step:\n" + step_by_step(q))
t = with_calculator(q)
print("\nwith a calculator:", t)

real answer:    33889284
answer only:    33907224

step by step:
To find the product of 6583 and 5148, you can use long multiplication or a calculator. Here's how to do it step-by-step:

1. Multiply 6583 by 8 (the units digit of 5148):
   - 3 * 8 = 24 (write down 4)
   - 8 * 8 = 64 (write down 4, carry over 6)
   - 5 * 8 = 40 + 6 = 46 (write down 6, carry over 4)
   - 6 * 8 = 48 + 4 = 52 (write down 2, carry over 5)

   Result so far: 462

2. Now multiply 6583 by 4 (the tens digit of 5148) and add the carried-over 4:
   - 3 * 4 = 12 (write down 2, carry over 1)
   - 8 * 4 = 32 + 1 = 33 (write down 3, carry over 3)
   - 5 * 4 = 20 + 3 = 23 (write down 3, carry over 2)
   - 6 * 4 = 24 + 2 = 26 (write down 6, carry over 2)

   Result so far: 2632

3. Finally, multiply 6583 by 5 (the hundreds digit of 5148) and add all the carried-over numbers:
   - 3 * 5 = 15 (write down 5, carry over 1)
   - 8 * 5 = 40 + 1 = 41 (write down 1, carry over 4)
   - 5 * 5 = 25 + 4 = 29 (write down 9, carry ov

In [8]:
# The check: does the number in its final sentence match what the calculator returned?
def numbers_in(text):
    return [int(n.replace(",", "")) for n in re.findall(r"\d[\d,]*", text)]

if t["used_calculator"]:
    said = numbers_in(t["final"])
    print("calculator returned:", t["calculator"], "· its sentence says:", said[-1] if said else None,
          "· match" if said and said[-1] == t["calculator"] else "· DOES NOT MATCH")
else:
    print("No calculator was used, so the answer is a guess.")

calculator returned: 33889284 · its sentence says: 33889284 · match
